# PaddleOCR-VL 1.6 — analyse de document par modèle vision-langage (0,9 B)

PaddleOCR-VL détecte d'abord la mise en page (titres, paragraphes, tableaux…), puis lit chaque bloc avec un petit
modèle vision-langage. Le malgache ne figure pas dans la liste des 109 langues de la version d'origine : la
validation visuelle ci-dessous permet de juger sur vos propres documents.
Démo en ligne : https://huggingface.co/spaces/PaddlePaddle/PaddleOCR-VL-1.6_Online_Demo

| | |
|---|---|
| Modèle | [PaddlePaddle/PaddleOCR-VL-1.6](https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6) |
| Licence | Apache-2.0 |
| Taille | ≈ 0,9 B paramètres (+ modèle de layout) |
| Matériel | GPU recommandé ; CPU possible mais lent (plusieurs dizaines de s / page). Python 3.8 – 3.12 |
| Sortie | Markdown (texte, tableaux, formules) + blocs avec boîte et type |
| PDF | pages rastérisées avec `pypdfium2` (fonction commune `load_document`) |

Aucune clé d'API ni token HuggingFace n'est nécessaire : les poids publics sont téléchargés au premier lancement
(dans `~/.cache/huggingface`). Le notebook se lance depuis le dossier `hf_ocr_notebooks/` (il importe `ocr_common.py`).

In [ ]:
%pip install -q paddlepaddle==3.2.1 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
%pip install -q "paddleocr[doc-parser]" pypdfium2 pillow pandas rapidfuzz datasets matplotlib

## 1. Jeux de données

Toutes les sources écrivent dans `data/<source>/` une image par échantillon et, à côté, sa vérité terrain `.txt` :

| Source | Contenu |
|---|---|
| `synthetic/` | 6 images + 1 PDF (français, anglais, tableau, petite police, rotation, bruit) |
| `malagasy/` | 5 images de phrases malgaches intégrées (sans réseau) |
| `hf_Lo-Renz-O_malagasy-sentence/` | phrases malgaches réelles du Hub, rendues en images (bruit / inclinaison) |
| `hf_Teklia_IAM-line/` | vraies images de lignes manuscrites anglaises (IAM) |
| `custom/` | vos documents (image ou PDF, + `nom.txt` optionnel pour mesurer le CER) |

Les datasets du Hub sont téléchargés une seule fois (paquet `datasets`) ; sans réseau, ils sont ignorés.

In [ ]:
import time
from pathlib import Path

from IPython.display import display

import ocr_common as oc

oc.make_test_set()
oc.make_malagasy_set()
for loader in (oc.load_hf_text_dataset, oc.load_hf_image_dataset):
    try:
        loader()
    except Exception as e:  # pas de réseau / pas de paquet datasets
        print(f"{loader.__name__} ignoré : {type(e).__name__}: {e}")

FILES = oc.list_inputs()
print(f"{len(FILES)} fichiers :", ", ".join(f"{f.parent.name}/{f.name}" for f in FILES))
oc.show_dataset([f for f in FILES if f.suffix != ".pdf"])

In [ ]:
import sys

import numpy as np

try:
    import paddle
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"paddlepaddle absent : aucune wheel pour Python {sys.version_info.major}.{sys.version_info.minor}. "
        "Relancez ce notebook dans un environnement Python 3.8 – 3.12."
    ) from e

DEVICE = "gpu" if paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0 else "cpu"

# Sur GPU NVIDIA, remplacez la 1re ligne d'installation par :
# %pip install paddlepaddle-gpu==3.2.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
print("Matériel :", oc.hardware(), "| DEVICE =", DEVICE)

## 2. Chargement du modèle

In [ ]:
t0 = time.perf_counter()
from paddleocr import PaddleOCRVL

pipeline = PaddleOCRVL(pipeline_version="v1.6", device=DEVICE)
LOAD_S = time.perf_counter() - t0
print(f"Modèle chargé en {LOAD_S:.1f}s sur {DEVICE}")

## 3. Fonction d'extraction


Le texte renvoyé est le Markdown de la page ; les boîtes sont celles des blocs de mise en page (`parsing_res_list`).

In [ ]:
def predict(image):
    res = pipeline.predict(np.array(image)[:, :, ::-1].copy())[0]
    blocks = res.json["res"]["parsing_res_list"]
    return res.markdown["markdown_texts"], [b["block_bbox"] for b in blocks]


def extract(image):
    return predict(image)[0]

## 4. Validation visuelle

Pour chaque image : à gauche la page avec les zones détectées (en rouge), à droite la vérité terrain et le texte lu,
avec le CER dans le titre. Commencez par une image malgache, puis parcourez tout le jeu de test.

In [ ]:
img = oc.load_document(oc.MALAGASY_DIR / "mg_vaovao.png")[0]
t = time.perf_counter()
text, boxes = predict(img)
print(f"({time.perf_counter() - t:.1f}s)\n{text}")
oc.show_result(img, text, boxes, ref=oc.ground_truth(oc.MALAGASY_DIR / "mg_vaovao.png"), title="mg_vaovao.png")

In [ ]:
# toutes les sources sauf les PDF ; augmentez max_files pour tout voir
oc.visual_check(predict, [f for f in FILES if f.suffix != ".pdf"], max_files=12)

### PDF multi-pages

In [ ]:
pdf_path = oc.SYNTH_DIR / "rapport_2_pages.pdf"
for i, page in enumerate(oc.load_document(pdf_path), 1):
    t = time.perf_counter()
    text = extract(page)
    print(f"--- page {i} ({time.perf_counter() - t:.1f}s)\n{text}\n")

## 5. Benchmark sur tout le jeu de test

Temps par page et précision (CER / WER) sur tous les fichiers. Le premier appel inclut un éventuel temps de compilation/« chauffe ».
Les textes extraits sont enregistrés dans `results/<modèle>/` et les mesures dans `results/<modèle>.csv`
(utilisées par le notebook `00_comparaison.ipynb`).

In [ ]:
MODEL_NAME = "PaddleOCR-VL-1.6"
df = oc.run_benchmark(extract, MODEL_NAME, FILES)
oc.save_meta(MODEL_NAME, repo="PaddlePaddle/PaddleOCR-VL-1.6", load_s=LOAD_S, device=str(DEVICE))
display(oc.summarize(df).to_frame(MODEL_NAME))

In [ ]:
display(df[[c for c in ("file", "type", "pages", "s_per_page", "cer", "wer", "error") if c in df]].round(3))
mg = df[df.file.str.startswith("mg_") | df.file.str.fullmatch(r"\d{3}\.png")]
if "cer" in mg and len(mg):
    print(f"CER moyen sur les échantillons malgaches et HF : {mg.cer.mean():.1%}")
oc.show_comparison(df, n=3)

## 6. Utiliser le modèle sur vos propres fichiers

Remplacez le chemin ci-dessous par une image ou un PDF ; le texte est enregistré à côté du fichier (`.ocr.txt`).

In [ ]:
def extract_file(path, dpi=200, max_pages=None):
    path = Path(path)
    pages = oc.load_document(path, dpi=dpi, max_pages=max_pages)
    text = "\n\n".join(extract(p) for p in pages)
    path.with_suffix(".ocr.txt").write_text(text, encoding="utf-8")
    return text


# print(extract_file("data/custom/mon_document.pdf"))